# Structures and quantities: a simply supported beam and earthwork volumes

> **Disclaimer.** CivKit results support, and do not replace, the judgement of a qualified engineer.
>
> Everything here runs offline: the `civkit` package makes no network calls.

In [ ]:
import civkit

## Simply supported beam

A UDL and a point load (Udoeyo, *Structural Analysis*, Example 7.2), in US customary units. Signs follow the cited
sources: a downward load gives negative deflection and slope.

**Conditional requirement:** `point_load_position` is optional, but required when `point_load > 0`.

In [ ]:
b = civkit.structural.beam.simply_supported(
    span="8 ft", udl="2 kip/ft", point_load="10 kip", point_load_position="4 ft",
    elastic_modulus="10000 ksi", second_moment="1000 in^4", position="4 ft",
    output_units={"reaction_a": "kip", "reaction_b": "kip", "deflection": "in", "end_slope_a": "rad"},
)
for name, q in b.outputs.items():
    print(f"{name:12} {q.value:.6g} {q.unit}")

In [ ]:
try:
    civkit.structural.beam.simply_supported(
        span="8 ft", point_load="10 kip", elastic_modulus="10000 ksi", second_moment="1000 in^4", position="4 ft"
    )
except civkit.CivKitError as e:
    print(e.code, e.input, "-", e.message)

## Earthwork volumes by average end area

Cut and fill areas at each chainage; one volume per segment between consecutive chainages, and totals. The `method`
is a choice: `average_end_area` everywhere, or `pyramid_at_zero_end`, which uses A·L/3 where one end area is zero.

In [ ]:
section = dict(
    chainages=([0, 50, 100, 150], "m"),
    cut_areas=([40, 42, 19, 34], "m^2"),
    fill_areas=([0, 0, 0, 0], "m^2"),
)
e = civkit.quantities.earthwork.average_end_area(method="average_end_area", **section)
for k, q in enumerate(e.outputs["cut_volumes"], start=1):
    print(f"segment {k}: {q.value:.1f} {q.unit}")
print("total cut", e.outputs["total_cut"].value, e.outputs["total_cut"].unit)

In [ ]:
wedge = dict(chainages=([0, 50], "m"), cut_areas=([0, 30], "m^2"), fill_areas=([0, 0], "m^2"))
for method in ["average_end_area", "pyramid_at_zero_end"]:
    v = civkit.quantities.earthwork.average_end_area(method=method, **wedge).outputs["cut_volumes"][0]
    print(f"{method:20} {v.value:.1f} {v.unit}")